# Spline Statistics
We consider the realization of a random periodic polynomial spline of a specified period and degree; its samples at the integers follow a distribution that can be specified, too. The ``Randomize`` buttom allow one to draw other realizations. We then print a few statistics from the current realization, either determined from the continuously defined spline, or empirically from its integer samples.

Finally, we plot in <span style="color:#1f77b4">**blue**</span> the realization of the spline, in <span style="color:#0343df">**bright blue**</span> its true average, in <span style="color:#00ffff">**bright cyan**</span> the bounds of its $\pm1$ standard deviation, in <span style="color:#d62728">**red**</span> the infimum of the image of the spline, and in <span style="color:#2ca02c">**green**</span> its supremum.

In [ ]:
# Load the required libraries
import ipywidgets as widgets
import math
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
max_period = 50 # Maximal period
max_degree = 5 # Maximal spline degree
plotpoints = 300 + 1 # Number of plot points
plotrange = sk.interval.Closed((-5.0, 5.0)) # Frozen display image
rng = np.random.default_rng() # Random number generator

# Display component
out = widgets.Output()

def plot_random_data (
    period,
    degree
):
    # Random spline samples
    f = np.zeros(period)
    if 0 == distribution_radiobuttons_widget.value: # Uniform
        f = rng.uniform(-math.sqrt(3.0), math.sqrt(3.0), period)
    elif 1 == distribution_radiobuttons_widget.value: # Gaussian
        f = rng.standard_normal(period)
    elif 2 == distribution_radiobuttons_widget.value: # Cauchy
        f = rng.standard_cauchy(period)
    # Sampled statistics
    empirical_mean = np.mean(f)
    empirical_stddev = np.std(f, ddof = 1)
    empirical_lb = np.min(f)
    empirical_ub = np.max(f)
    # Random periodic spline
    s = sk.PeriodicSpline1D.from_samples(f, degree = degree)
    # Continuous statistics
    avg = s.mean() # Average of this realization
    stddev = math.sqrt(s.variance()) # Standard deviation of this realization
    image = s.image() # Image of this realization
    lb = image.infimum # Lower bound of the image
    ub = image.supremum # Upper bound of the image
    out.clear_output()
    with out:
        print("")
        print("Continuous Statistics")
        print("---------------------")
        print("True average = {}".format(avg))
        print("True standard deviation = {}".format(stddev))
        print("True infimum = {}".format(lb))
        print("True supremum = {}".format(ub))
        print("")
        print("Sampled Statistics")
        print("------------------")
        print("Empirical mean = {}".format(empirical_mean))
        print("Unbiased empirical standard deviation = {}".format(empirical_stddev))
        print("Minimal sampled value = {}".format(empirical_lb))
        print("Maximal sampled value = {}".format(empirical_ub))
        print("")
        subplot = plt.subplots()
        # Spline
        s.plot(
            subplot,
            plotpoints = plotpoints,
            plotrange = plotrange,
            knot_marker = " "
        )
        # Average
        plt.plot([-1.0, period + 1.0], [avg, avg], "-b", lw = 0.5)
        # Average - standard deviation
        plt.plot([-1.0, period + 1.0], [avg - stddev, avg - stddev], "-c", lw = 0.5)
        # Average + standard deviation
        plt.plot([-1.0, period + 1.0], [avg + stddev, avg + stddev], "-c", lw = 0.5)
        # Lower bound
        plt.plot([-1.0, period + 1.0], [lb, lb], "-C3", lw = 0.5)
        # Upper bound
        plt.plot([-1.0, period + 1.0], [ub, ub], "-C2", lw = 0.5)
        plt.show()

# Period slider
period_intslider_widget = widgets.IntSlider(
    value = 20,
    min = 2,
    max = max_period,
    description = "period"
)
def on_period_changed (
    change
):
    plot_random_data(period_intslider_widget.value, degree_intslider_widget.value)
period_intslider_widget.observe(on_period_changed, names = "value")

# Degree slider
degree_intslider_widget = widgets.IntSlider(
    value = 3,
    min = 0,
    max = max_degree,
    description = "degree"
)
def on_degree_changed (
    change
):
    plot_random_data(period_intslider_widget.value, degree_intslider_widget.value)
degree_intslider_widget.observe(on_degree_changed, names = "value")

# Selection of the distribution
distribution_radiobuttons_widget = widgets.RadioButtons(
    options = [("Uniform", 0), ("Gaussian", 1), ("Cauchy", 2)],
    value = 0,
    description = "Distribution of the Samples:"
)
def on_distribution_changed (
    change
):
    plot_random_data(period_intslider_widget.value, degree_intslider_widget.value)
distribution_radiobuttons_widget.observe(on_distribution_changed, names = "value")

# Randomization button
randomize_button_widget = widgets.Button(description = "Randomize")
def on_randomize_button_clicked (
    b
):
    plot_random_data(period_intslider_widget.value, degree_intslider_widget.value)
randomize_button_widget.on_click(on_randomize_button_clicked)

# Layout
display(
    period_intslider_widget,
    degree_intslider_widget,
    distribution_radiobuttons_widget,
    randomize_button_widget,
    out
)
plot_random_data(period_intslider_widget.value, degree_intslider_widget.value)
